# BNCC Machine Learning Workshop: Regression

Pada notebook ini, kita akan mempelajari alur kerja machine learning untuk masalah regresi secara end-to-end. Tujuannya adalah membangun model yang dapat memprediksi harga rumah berdasarkan fitur-fitur yang tersedia di dataset.

Kita akan mulai dari memahami data, membersihkan data, melakukan feature engineering, menyiapkan preprocessing, melatih model, lalu mengevaluasi hasil prediksi.

In [ ]:
print("Welcome to BNCC Machine Learning Workshop - Regression")

Sebelum membangun model, kita perlu mengetahui sumber data yang digunakan. Dataset pada notebook ini berisi data harga rumah di Jakarta beserta beberapa fitur seperti lokasi, jumlah kamar, luas tanah, dan luas bangunan.

Informasi sumber dataset penting supaya analisis kita bisa ditelusuri kembali dan peserta memahami konteks data yang sedang dipelajari.

Link Dataset: https://www.kaggle.com/datasets/abiyyurasyiq/jakarta-house-price-dataset/data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

## Memuat Dataset

Pada bagian ini, kita mulai memuat file dataset ke dalam pandas DataFrame. Setelah data berhasil dibaca, tampilkan beberapa baris pertama dan memastikan kolom-kolomnya sudah sesuai.

In [ ]:
# TODO: lengkapi kode berikut untuk memuat dataset

...

Informasi Fitur Dataset:

1. index = Unique identifier for each data record
2. price = House price (Rupiah)
3. district = District (kecamatan) where the house is located
4. city =	City where the house is located
5. bed_rooms = Number of bedrooms
6. bath_rooms = Number of bathrooms
7. carport = Number of carports
8. land_area = Land area of the house (in square meters)
9. building_area = Building area of the house (in square meters)

Cek struktur dataset: jumlah baris, nama kolom, tipe data, dan apakah ada missing value.

In [ ]:
# TODO: lengkapi kode berikut untuk melihat struktur dataset

...

Cek ringkasan statistik kolom numerik seperti rata-rata, median, nilai minimum, maksimum, dan sebaran data.

In [ ]:
# TODO: lengkapi kode berikut untuk melihat deskripsi dataset

...

Angka pada dataset kadang ditampilkan dalam format scientific notation, terutama untuk nilai yang sangat besar seperti harga rumah.

Pahami cara membaca format ini supaya tidak keliru saat menginterpretasikan hasil statistik atau output model.

In [ ]:
print("Cara membaca format angka scientific, contoh: 2 x 1e+10 = 2 x 10^10 (10,000,000,000)")

print(f"Apakah 2 x 1e+10 = 2 x 10,000,000,000 ?")

print(2 * 1e+10 == 2 * 10_000_000_000)

Data duplikat bisa membuat model belajar dari baris yang sama lebih dari sekali, baiknya kita hilangkan data redundant tersebut.

Cek apakah ada baris duplikat, lalu bersihkan jika diperlukan.

In [ ]:
# TODO: lengkapi kode berikut untuk hilangkan data duplikat di dataset

...

## EDA (Exploratory Data Analysis)

Sebelum membuat model, cek sebaran fitur kategori seperti lokasi rumah.

Visualisasi ini membantu melihat kategori mana yang paling sering muncul dan apakah distribusinya seimbang atau tidak. Nilai `JUMLAH_DISTRICT` dan `JUMLAH_CITY` bisa diganti sesuai jumlah kategori yang ingin ditampilkan pada figure.

In [ ]:
JUMLAH_DISTRICT = ...
JUMLAH_CITY = ...

print(f"Jumlah kategori pada kolom district adalah: {len(df['district'].unique())}")
print(f"Jumlah kategori pada kolom city adalah: {len(df['city'].unique())}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1
df["district"].value_counts().head(JUMLAH_DISTRICT).plot(
    kind="bar",
    ax=axes[0]
)
axes[0].set_xlabel("District")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribusi district")
axes[0].grid(axis="y", alpha=0.75, linestyle="--")

# Plot 2
df["city"].value_counts().head(JUMLAH_CITY).plot(
    kind="bar",
    ax=axes[1]
)
axes[1].set_xlabel("City")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribusi city")
axes[1].grid(axis="y", alpha=0.75, linestyle="--")

plt.tight_layout()
plt.show()

Beberapa kolom numerik bisa memiliki nilai ekstrem, sehingga histogram menjadi sulit dibaca karena sebagian besar data terlihat menumpuk di area kecil.

Gunakan flag `ENABLE_99_PERCENTILE_LIMIT` untuk memilih tampilan plot. Set menjadi `True` jika ingin membatasi x-axis sampai percentile tertentu, atau `False` jika ingin melihat seluruh range data.

In [ ]:
# TODO: lengkapi kode berikut untuk mengatur plot histogram

ENABLE_99_PERCENTILE_LIMIT = ...

# fitur numerik
columns = ["price", "bed_rooms", "bath_rooms", "carport", "land_area", "building_area"]

X_AXIS_PERCENTILE = 0.99

fig, axes = plt.subplots(4, 3, figsize=(10, 8))
axes = axes.flatten()

for ax, col in zip(axes, columns):
    ax.hist(df[col].dropna(), bins=50, edgecolor="black")

    if ENABLE_99_PERCENTILE_LIMIT:
        x_max = df[col].quantile(X_AXIS_PERCENTILE)
        ax.set_xlim(left=0, right=x_max)

    ax.set_title(col)
    ax.grid(alpha=0.2)

# remove unused axes
for ax in axes[len(columns):]:
    ax.remove()

plt.suptitle("Visualisasi Histogram dari fitur Numerik")
plt.tight_layout()

Bandingkan nilai quantile untuk melihat sebaran fitur numerik secara lebih ringkas.

Perhatikan jarak antara median, P95, P99, dan nilai maksimum. Jika jaraknya sangat jauh, fitur tersebut kemungkinan memiliki outlier atau distribusi yang sangat skewed.

In [ ]:
quantiles = [0.25, 0.50, 0.75, 0.95, 0.99, 1.0]
labels = ["Q1", "Median", "Q3", "P95", "P99", "Max"]

fig, axes = plt.subplots(4, 3, figsize=(10, 10))
axes = axes.flatten()

for ax, col in zip(axes, columns):
    values = df[col].quantile(quantiles)

    ax.bar(labels, values)

    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2)

for ax in axes[len(columns):]:
    ax.remove()

plt.suptitle("Perbandingan Range Numerik di masing-masing Quantile")
plt.tight_layout()
plt.show()

Box plot membantu membandingkan sebaran harga antar kota, terutama median, rentang utama data, dan nilai ekstrem.

Variabel `TAMPILKAN_OUTLIER` bisa diubah menjadi `True` atau `False`. Jika `True`, titik outlier akan ditampilkan. Jika `False`, outlier disembunyikan agar perbandingan antar box lebih mudah dibaca.

In [ ]:
# TODO: lengkapi kode berikut untuk mengatur plot outlier

TAMPILKAN_OUTLIER = ...

# menampilkan urutan berdasarkan rata rata harga di setiap kota
order = (
    df.groupby("city")["price"]
    .mean()
    .sort_values()
    .index
)

plt.figure(figsize=(8, 6))

sns.boxplot(
    x="city",
    y="price",
    data=df,
    order=order,
    width=0.5,
    hue="city",
    palette="Set2",
    legend=False,
    showfliers=TAMPILKAN_OUTLIER
)

plt.title("Distribusi harga rumah di setiap kota", fontsize=14)
plt.xticks(rotation=90)
plt.ylabel("Price", labelpad=30)
plt.xlabel("City")
plt.grid(True, axis="y", alpha=0.3)

plt.show()

Korelasi membantu melihat arah dan kekuatan hubungan linear antar fitur numerik. Pada cell ini, angka korelasi dihitung menggunakan Pearson correlation, yaitu perhitungan dari pasangan nilai pada dua kolom, misalnya bagaimana perubahan `land_area` berkaitan dengan perubahan `price`.

Nilai mendekati 1 berarti hubungan positif kuat, mendekati -1 berarti hubungan negatif kuat, dan mendekati 0 berarti hubungan linear lemah. Perhatikan fitur mana yang paling berkaitan dengan `price`, tapi ingat bahwa korelasi tidak selalu berarti sebab-akibat.

In [ ]:
# NOTE: hitung correlation matrix dari dataframe

corr = ...

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
)
plt.title("Korelasi Fitur Numerik")
plt.tight_layout()
plt.show()

Sebelum membuat scatter plot, lihat dulu daftar kolom yang tersedia agar bisa memilih fitur yang ingin dibandingkan dengan `price`.

Pada cell berikutnya, nama kolom yang dipilih akan digunakan sebagai sumbu x untuk melihat pola hubungan fitur tersebut dengan harga rumah.

In [ ]:
print(f"Kolom yang bisa di coba:\n\n {list(df.columns)}")

In [ ]:
# TODO: kolom yang dapat di isi berdasarkan kolom yang tersedia pada output di atas

X_COLUMN = ...

plt.figure(figsize=(10, 6))

# Scatter plot
sns.scatterplot(
    data=df,
    x=X_COLUMN,
    y="price",
    hue="city",
    alpha=0.6
)

# Linear regression line
sns.regplot(
    data=df,
    x=X_COLUMN,
    y="price",
    scatter=False,
    color="black"
)

plt.title(f"Hubungan {X_COLUMN} dengan harga")
plt.xlabel(X_COLUMN)
plt.ylabel("Price")
plt.grid(alpha=0.2)

plt.show()

## Membersihkan Data

Pada tahap membersihkan data, sebaiknya gunakan salinan dataset agar data asli tetap aman.

Dengan begitu, jika ada proses cleaning yang perlu diulang atau dibandingkan, kita masih punya versi awal dataset sebagai referensi.

In [ ]:
# TODO: copy dataframe terlebih dahulu supaya dataframe asli (df) tidak berubah

...

In [ ]:
print(f"Cek kolom yang perlu di buang:\n\n {list(df_clean.columns)}")

Tidak semua kolom berguna untuk analisis atau model. Kolom yang hanya berfungsi sebagai penanda urutan data biasanya tidak membawa informasi tentang harga rumah.

Hapus kolom yang tidak relevan, lalu cek kembali daftar kolom setelah proses tersebut.

In [ ]:
# TODO: buang kolom index, karena kolom tersebut tidak penting bagi model dan analisis

...

Missing value perlu dicek sebelum modeling karena beberapa algoritma tidak bisa menerima data kosong.

Hitung jumlah nilai kosong pada setiap kolom, lalu tentukan kolom mana yang perlu ditangani.

In [ ]:
# TODO: cek jumlah data kosong di setiap kolom

...

In [ ]:
# tampilkan data yang kosong
df_clean[
    df_clean["land_area"].isna() |
    df_clean["building_area"].isna()
]

Untuk modeling, missing value sebaiknya diisi setelah train-test split supaya statistik seperti median hanya dihitung dari data training.

In [ ]:
# kolom numerik yang memiliki missing value
columns_to_fill = ["land_area", "building_area"]

df_clean[columns_to_fill].isna().sum()

Ada beberapa cara menangani missing value. Isi dengan median, mean, atau mode jika kolom masih penting dan jumlah data kosongnya tidak terlalu besar.

Drop column bisa dipilih jika satu kolom terlalu banyak kosong atau tidak relevan. Drop row bisa dipilih jika jumlah baris yang kosong sangat sedikit dan tidak banyak mengurangi data.

Pada bagian ini, kita coba metode pandas terlebih dahulu untuk mengisi missing value dengan nilai median.

In [ ]:
# demonstrasi mengisi missing value dengan pandas fillna + median
# gunakan copy supaya df_clean untuk modeling tidak berubah sebelum train-test split

df_pandas_fill = df_clean.copy()

land_area_median = df_pandas_fill["land_area"].median()
df_pandas_fill["land_area"] = df_pandas_fill["land_area"].fillna(land_area_median)

# TODO: lengkapi median filling untuk kolom building area berdasarkan contoh pada kolom land area

...

df_pandas_fill[columns_to_fill].isna().sum()

Setelah mencoba cara manual dengan pandas, kita bandingkan dengan pendekatan dari scikit-learn.

Imputer dari scikit-learn lebih mudah digabungkan ke workflow modeling karena proses menghitung nilai pengisi dan menerapkannya bisa dipisahkan dengan jelas.

In [ ]:
from sklearn.impute import SimpleImputer

# copy data untuk demonstrasi cara kerja imputer
# ini belum dipakai untuk modeling
# pada modeling, imputer akan di-fit setelah train-test split
df_imputer = df_clean.copy()

imputer = SimpleImputer(strategy="median")

# lakukan fitting terhadap imputer = proses menghitung strategy
imputer.fit(df_imputer[columns_to_fill])

print(f"Kolom yang masuk kedalam imputer adalah: {imputer.feature_names_in_}")
print(f"Strategy filling yang digunakan adalah: {imputer.strategy}")

In [ ]:
print(f"Median yang dihasilkan imputer untuk fitur: {columns_to_fill} adalah:")
print(imputer.statistics_)
print()

print(f"Median yang dihasilkan pandas dataframe untuk fitur: {columns_to_fill} adalah:")
print(df_pandas_fill[columns_to_fill].median())

In [ ]:
# lakukan transform = proses filling terjadi
df_imputer[columns_to_fill] = imputer.transform(df_imputer[columns_to_fill])

In [ ]:
df_imputer[columns_to_fill].isna().sum()

## Eksperimen dengan kombinasi fitur (Feature Engineering)

Feature engineering membuat fitur baru dari kolom yang sudah ada agar model mendapat informasi yang lebih bermakna. Fitur tambahan bisa membantu model menangkap pola yang sebelumnya tidak terlihat langsung dari kolom asli.

Karena perhitungan di sini hanya memakai nilai dari baris yang sama, proses ini aman dilakukan sebelum train-test split. 

Buatkan fitur `total_rooms` merangkum jumlah kamar, sedangkan `building_land_ratio` membandingkan luas bangunan dengan luas tanah.

In [ ]:
# TODO: membuat fitur baru dari fitur yang sudah ada

...

In [ ]:
# TODO: cek missing value karena kita mengkombinasi missing value

...

## Train and Test Splitting

Sebelum membagi data, tentukan dulu mana kolom yang menjadi fitur dan mana kolom yang menjadi target.

Pada kasus regression ini, target adalah nilai yang ingin diprediksi, sedangkan fitur adalah informasi yang digunakan untuk membantu prediksi.

In [ ]:
# TODO: pisahkan data fitur dan target fitur

...

Setelah fitur dan target dipisahkan, data perlu dibagi menjadi data training dan data testing.

Data training digunakan untuk melatih model, sedangkan data testing digunakan untuk mengevaluasi performa model pada data yang belum pernah dilihat.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: lakukan splitting dan tentukan jumlah data yang akan di jadikan testing

...

## Mengisi missing menggunakan Imputer

Imputer dilakukan setelah train-test split supaya nilai pengisi seperti median hanya dihitung dari data training. Jika median dihitung dari seluruh dataset, test set ikut memengaruhi proses training dan evaluasi model bisa menjadi kurang jujur.

In [ ]:
from sklearn.impute import SimpleImputer

columns_to_fill = ["land_area", "building_area", "building_land_ratio"]

# TODO: lakukan filling terhadap missing value berdasarkan kolom yang sudah di tandai

...

## Mengubah fitur teks dan kategori

Model machine learning umumnya hanya bisa memproses data dalam bentuk angka, sehingga fitur kategori berbentuk teks perlu diubah menjadi nilai numerik terlebih dahulu.

Ordinal encoder mengubah setiap kategori menjadi angka tertentu. Teknik ini paling cocok digunakan ketika kategori memiliki urutan alami, misalnya rendah, sedang, tinggi. Pada fitur seperti `district`, ordinal encoding bisa digunakan sebagai pendekatan sederhana, tetapi perlu diingat bahwa angka hasil encoding tidak selalu berarti ada urutan kualitas antar lokasi.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# TODO: lakukan encoding terhadap kolom `district` menggunakan ordinal encoder

...

Kita dapat melihat pasangan kategori beserta angka encoded dari masing masing kategori dari kolom district.

In [ ]:
mapping_df = pd.DataFrame({
    "district": ordinal_encoder.categories_[0],
    "encoded_value": range(len(ordinal_encoder.categories_[0]))
})

mapping_df.head(10)

One-hot encoding digunakan untuk mengubah fitur kategori menjadi beberapa kolom angka berisi 0 atau 1.

Teknik ini cocok digunakan ketika kategori tidak memiliki urutan alami, seperti nama kota. Dengan one-hot encoding, model tidak menganggap satu kategori lebih tinggi atau lebih rendah dari kategori lain.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# TODO: lakukan encoding terhadap kolom `city` menggunakan one hot encoder

...

Hasil one-hot encoding berbentuk array atau dataframe baru dengan beberapa kolom tambahan, satu kolom untuk setiap kategori.

Berbeda dengan ordinal encoding yang langsung mengganti isi satu kolom, hasil one-hot encoding perlu digabungkan kembali ke dataframe utama agar bisa digunakan bersama fitur lainnya saat training model.

In [ ]:
train_city_encoded = pd.DataFrame(
    train_city_encoded,
    columns=city_columns,
    index=X_train.index
)

test_city_encoded = pd.DataFrame(
    test_city_encoded,
    columns=city_columns,
    index=X_test.index
)

In [ ]:
# gabungkan ke df_clean

X_train = pd.concat(
    [
        X_train.drop(columns="city"),
        train_city_encoded
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test.drop(columns="city"),
        test_city_encoded
    ],
    axis=1
)

X_train.head()

## Feature Scaling pada fitur Numerik

Feature scaling adalah proses menyamakan skala nilai antar fitur numerik agar tidak ada fitur yang terlalu mendominasi hanya karena angkanya lebih besar.

Scaling penting terutama untuk model yang sensitif terhadap jarak atau besar kecilnya nilai fitur. Dengan scaling, fitur seperti `land_area`, `building_area`, dan jumlah ruangan bisa dibandingkan dalam skala yang lebih seimbang.

In [ ]:
X_train.info()

In [ ]:
scaling_columns = [
    "bed_rooms",
    "bath_rooms",
    "carport",
    "land_area",
    "building_area",
    "total_rooms",
    "building_land_ratio",
]

In [ ]:
unscaled_data = X_train[scaling_columns]
unscaled_data.describe()

Ada beberapa teknik scaling yang bisa digunakan, tergantung karakteristik data dan model yang dipakai.

- Standard Scaler: cocok ketika fitur numerik memiliki skala berbeda dan data tidak perlu dibatasi ke range tertentu. Teknik ini mengubah nilai berdasarkan rata-rata dan standar deviasi.
- MinMax Scaler: cocok ketika kita ingin mengubah nilai ke range tertentu, biasanya 0 sampai 1. Teknik ini lebih sensitif terhadap outlier karena nilai minimum dan maksimum sangat memengaruhi hasil scaling.

Pada dataset ini, beberapa fitur memiliki outlier dan distribusi yang cukup skewed. Karena itu, kita menggunakan Standard Scaler agar skala fitur lebih seimbang tanpa memaksa semua nilai masuk ke range 0 sampai 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: lakukan scaling terhadap kolom numerik menggunakan standard scaler

...

In [ ]:
X_train[scaling_columns].describe()

## Training menggunakan Model Linear Regression

Setelah data selesai diproses, kita bisa mulai melatih model regression.

Linear Regression cocok digunakan sebagai baseline model karena sederhana, cepat dilatih, dan mudah diinterpretasikan. Dalam machine learning, kita sebaiknya mulai dari baseline terlebih dahulu agar punya pembanding sebelum mencoba model yang lebih kompleks.

In [ ]:
from sklearn.linear_model import LinearRegression

# NOTE: lakukan training model menggunakan linear regression dengan training data

...

## Evaluasi Model Linear Regression

Evaluasi pada training set digunakan untuk melihat seberapa baik model mempelajari data yang digunakan saat training.

Nilai ini nantinya perlu dibandingkan dengan performa pada test set. Jika performa training jauh lebih baik daripada test, model bisa jadi mengalami overfitting, yaitu terlalu menyesuaikan diri dengan data training tetapi kurang baik saat menghadapi data baru.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TODO: buatkan fungsi untuk menghitung metriks regression

...

In [ ]:
# TODO: evaluasi performa model linear regression pada training set

...

Setelah melihat performa pada training set, evaluasi juga perlu dilakukan pada test set.

Test set berisi data yang tidak digunakan saat training, sehingga hasil evaluasi ini lebih menggambarkan kemampuan model pada data baru. Bandingkan metrik training dan test untuk melihat apakah model cukup general atau mulai menunjukkan tanda overfitting.

In [ ]:
# TODO: lakukan prediksi model linear regression dengan test set beserta evaluasi nya

...

Disini kita bisa membuat figure plot dari hasil prediksi model linear regression.

Plot pertama membandingkan harga aktual dengan harga prediksi. Semakin dekat titik-titik ke garis diagonal, semakin dekat prediksi model dengan nilai aslinya.

Plot kedua adalah residual plot, yaitu selisih antara nilai aktual dan prediksi. Residual yang tersebar acak di sekitar garis 0 menunjukkan error yang lebih stabil. Jika terlihat pola tertentu, model mungkin belum menangkap hubungan data dengan baik.

In [ ]:
# Residual = nilai aktual - nilai prediksi
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Scatter Plot: Actual vs Predicted Price
sns.scatterplot(
    x=y_test,
    y=y_pred,
    alpha=0.5,
    ax=axes[0]
)
# Garis prediksi sempurna: y = x
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

axes[0].plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)
axes[0].set_title("Actual vs Predicted Price")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
axes[0].grid(alpha=0.2)

# 2. Residual Plot
sns.scatterplot(
    x=y_pred,
    y=residuals,
    alpha=0.5,
    ax=axes[1]
)
# Garis residual = 0
axes[1].axhline(
    y=0,
    linestyle="--"
)
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted Price")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Membuat Pipeline versi ColumnTransformers

Preprocessing sebelumnya dibuat secara manual agar setiap langkah lebih mudah dipahami.

Namun dalam workflow machine learning yang lebih rapi, preprocessing dan model sebaiknya digabungkan ke dalam pipeline. Dengan pipeline dan ColumnTransformer, proses seperti imputation, encoding, scaling, feature engineering, dan training bisa dijalankan dalam urutan yang konsisten. Ini juga mengurangi risiko lupa menerapkan preprocessing yang sama pada data training, test, atau data baru saat prediksi.

In [ ]:
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["total_rooms"] = X["bed_rooms"] + X["bath_rooms"]
        X["building_land_ratio"] = X["building_area"] / X["land_area"]

        return X


class HousePriceModel:
    def __init__(self, model=None):
        if model is None:
            model = LinearRegression()

        self.model = model 

        self.base_numeric_features = [
            "bed_rooms",
            "bath_rooms",
            "carport",
            "land_area",
            "building_area"
        ]
        self.engineered_numeric_features = [
            "total_rooms",
            "building_land_ratio"
        ]
        self.numeric_features = self.base_numeric_features + self.engineered_numeric_features
        self.ordinal_features = ["district"]
        self.onehot_features = ["city"]

        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        district_pipeline = Pipeline([
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ])
        city_pipeline = Pipeline([
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])

        preprocessor = ColumnTransformer([
            ("numeric", numeric_pipeline, self.numeric_features),
            ("district", district_pipeline, self.ordinal_features),
            ("city", city_pipeline, self.onehot_features)
        ])

        self.pipeline = Pipeline([
            ("feature_engineering", FeatureEngineer()),
            ("preprocessor", preprocessor),
            ("model", self.model)
        ])

    def train(self,X_train,y_train):
        self.pipeline.fit(X_train,y_train)
        return self

    def tune(
        self,
        X_train,
        y_train,
        param_grid,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    ):
        search = GridSearchCV(
            estimator=self.pipeline,
            param_grid=param_grid,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs
        )

        search.fit(X_train, y_train)

        self.pipeline = search.best_estimator_

        return {
            "best_params": search.best_params_,
            "best_score": search.best_score_
        }

    def evaluate(self, X, y):
        y_pred = self.pipeline.predict(X)

        return calculate_metrics(y, y_pred)

    def predict(self, X):
        return self.pipeline.predict(X)

Pada versi pipeline, kita mulai lagi dari data mentah agar seluruh preprocessing dilakukan di dalam pipeline.

Fitur yang dipilih adalah kolom asli dari dataset, sedangkan feature engineering, imputation, encoding, dan scaling akan ditangani otomatis oleh pipeline.

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset/jakarta_house.csv")

X = df[["district", "city", "bed_rooms", "bath_rooms", "carport", "land_area", "building_area"]]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

Model baseline yang sama digunakan kembali, tetapi kali ini dibungkus dalam class pipeline.

Dengan cara ini, training dan evaluasi bisa dilakukan lebih ringkas, sementara urutan preprocessing tetap konsisten untuk training set maupun test set.

In [ ]:
from sklearn.linear_model import LinearRegression

USE_MODEL = LinearRegression()

house_model = HousePriceModel(model=USE_MODEL)

house_model.train(X_train, y_train)

print(f"Metriks evaluasi pada training set: {house_model.evaluate(X_train, y_train)}")
print(f"Metriks evaluasi pada test set: {house_model.evaluate(X_test, y_test)}")

## Melakukan Prediksi dengan data rumah baru

Setelah pipeline dilatih, kita bisa menggunakannya untuk memprediksi harga rumah baru.

Ubah nilai input pada `new_house` sesuai skenario yang ingin dicoba, misalnya mengganti lokasi, jumlah kamar, luas tanah, atau luas bangunan. Pastikan nama kolom tetap sama seperti fitur yang digunakan saat training.

In [ ]:
# NOTE: ubah input untuk mencoba prediksi

new_house = pd.DataFrame({
    "district": ["Tebet"],
    "city": ["Jakarta Selatan"],
    "bed_rooms": [4],
    "bath_rooms": [3],
    "carport": [2],
    "land_area": [150],
    "building_area": [200]
})

prediksi = house_model.predict(new_house)[0]

print(f"harga prediksi rumah sesuai input adalah: {prediksi:,.0f} rupiah")

## Eksperimen dengan Model lain dan Hyperparameter Tuning

Dalam machine learning, kita biasanya tidak berhenti pada satu model saja. Model yang berbeda bisa menangkap pola data dengan cara yang berbeda, sehingga kita perlu membandingkan beberapa model untuk melihat mana yang paling cocok dengan dataset dan tujuan prediksi.

Hyperparameter tuning adalah proses mencoba beberapa kombinasi pengaturan model sebelum training, lalu memilih kombinasi yang memberikan performa terbaik. Berbeda dengan parameter yang dipelajari model saat training, hyperparameter ditentukan terlebih dahulu oleh kita.

In [ ]:
from sklearn.linear_model import Ridge

USE_MODEL = Ridge()

house_model = HousePriceModel(model=USE_MODEL)

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

result = house_model.tune(
    X_train,
    y_train,
    param_grid=param_grid,
    cv=5
)

print(result)
print()

print(f"Metriks evaluasi pada training set: {house_model.evaluate(X_train, y_train)}")
print(f"Metriks evaluasi pada test set: {house_model.evaluate(X_test, y_test)}")

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

USE_MODEL = KNeighborsRegressor()

house_model = HousePriceModel(model=USE_MODEL)

param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"]
}

result = house_model.tune(
    X_train,
    y_train,
    param_grid=param_grid,
    cv=5
)

print(result)
print()

print(f"Metriks evaluasi pada training set: {house_model.evaluate(X_train, y_train)}")
print(f"Metriks evaluasi pada test set: {house_model.evaluate(X_test, y_test)}")

In [ ]:
print("Congratulations, Regression is Finished!")